# *MICE for CATS* Replication Implementation for Feature Extraction
This python notebook replicates the methods in the *MICE for Cats* paper but modifies the code to fit in the application of Abstain/Clarify/Answer scenario.

### Setup

In [1]:
import torch
import pandas as pd
import os
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset, Dataset, load_from_disk
from bert_score import BERTScorer
from tqdm import tqdm
from huggingface_hub import login

/home/brandon/MICEforCATS-Extension/local_execute/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# env setup
MODEL_NAME = "meta-llama/Meta-Llama-3-8B-Instruct"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

from pathlib import Path
parent_dir = Path.cwd().parent
BASE_PATH = parent_dir

DATA_PATH = f"{BASE_PATH}/original_nb_data/Data/"
OUTPUT_PATH = f"{BASE_PATH}/original_nb_data/MICE_Output/"

MAX_NEW_TOKENS = 100

In [3]:
# Papermill parameters — overridable via `-p NAME VALUE` on the command line.
LIMIT = None  # None for full run, integer for test runs

In [4]:
# Parameters
LIMIT = 3


In [5]:
# huggingface authentication for LLM access
# make sure you save your HF token in .env in the same directory
from dotenv import load_dotenv

load_dotenv()
token = os.getenv("HF_TOKEN")

In [6]:
# load model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

model.eval()

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Fetching 4 files:   0%|                                                                           | 0/4 [00:00<?, ?it/s]

Fetching 4 files:  25%|████████████████▌                                                 | 1/4 [03:28<10:24, 208.21s/it]

Fetching 4 files: 100%|███████████████████████████████████████████████████████████████████| 4/4 [03:28<00:00, 52.05s/it]

Loading weights:   0%|                                                                          | 0/291 [00:00<?, ?it/s]

...

Loading weights:  98%|██████████████████████████████████████████████████████████████▍ | 284/291 [00:09<00:00, 38.34it/s]

Loading weights:  99%|███████████████████████████████████████████████████████████████▌| 289/291 [00:09<00:00, 40.34it/s]

Loading weights: 100%|████████████████████████████████████████████████████████████████| 291/291 [00:09<00:00, 30.97it/s]

Some parameters are on the meta device because they were offloaded to the cpu and disk.


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((4096,), eps=1e-05)
  

In [7]:
# initiate BERTScore from library
scorer = BERTScorer(
    lang="en",
    device=DEVICE,
    rescale_with_baseline=True
)

Loading weights:   0%|                                                                          | 0/389 [00:00<?, ?it/s]

Loading weights:  44%|███████████████████████████▍                                  | 172/389 [00:00<00:00, 1718.11it/s]

Loading weights: 100%|██████████████████████████████████████████████████████████████| 389/389 [00:00<00:00, 2043.63it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [8]:
# define features extraction
def extract_features(prompt):

    # format the input
    messages = [{"role": "user", "content": prompt}]
    formatted = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)

    # save length of prompt for later calcs
    prompt_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        generated_output = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            output_hidden_states=True,
            output_scores=True,
            return_dict_in_generate=True
        )

    sequences = generated_output.sequences
    hidden_states = generated_output.hidden_states  # (gen_steps, layers, batch, seq_len_t, dim)
    scores = generated_output.scores                # (gen_steps, batch, vocab)
    generated_tokens = sequences[0, prompt_len:]   # (gen_len,)

    # final reference text
    final_text = tokenizer.decode(generated_tokens, skip_special_tokens=True)


    lm_head = model.lm_head
    norm = model.model.norm

    # intermediate layer tokens
    layer_texts_per_token = []
    for step_hidden in hidden_states:
        step_layer_tokens = []
        for layer_h in step_hidden[1:-1]:           # skip embed and final layer
            logit = lm_head(norm(layer_h))          # (batch, 1, vocab)
            token = torch.argmax(logit, dim=-1)[0, -1]  # (1,)
            step_layer_tokens.append(token.item())
        layer_texts_per_token.append(step_layer_tokens)

    # decode intermediate layer texts
    num_layers = len(layer_texts_per_token[0])
    layer_texts = []
    for layer_idx in range(num_layers):
        token_ids = [layer_texts_per_token[step][layer_idx]
                     for step in range(len(hidden_states))]
        text = tokenizer.decode(token_ids, skip_special_tokens=True)
        layer_texts.append(text if text.strip() else "[EMPTY]")

    # rm: BERT scores
    # rm: precision, recall, f1 = calculate_bertscore(layer_texts, final_text)

    # raw confidence (log confidence calc)
    token_probs = torch.stack([
        torch.softmax(s, dim=-1) for s in scores
    ], dim=1)  # (batch, gen_len, vocab)

    token_probs_generated = token_probs[0].gather(1, generated_tokens.unsqueeze(-1)).squeeze(-1)  # (gen_len,)

    # sanity check
    assert token_probs_generated.shape[0] == len(generated_tokens), \
        f"Shape mismatch: {token_probs_generated.shape[0]} vs {len(generated_tokens)}"

    log_confidence = torch.sum(torch.log(token_probs_generated + 1e-10)).item()

    # nomalized log confidence, use this
    norm_log_confidence = log_confidence / len(generated_tokens)

    return {
        "final_text": final_text,
        "layer_texts": layer_texts,
        # rm: "precision": precision,
        # rm: "recall": recall,
        # rm: "f1": f1,
        "log_confidence": log_confidence,
        "normalized_log_confidence": norm_log_confidence
    }

### Extract Features for Prompts

In [9]:
# running feature extraction on 500 questions, save every once a while so not too much lost progress if dc
import pickle

df_input = pd.read_csv(os.path.join(DATA_PATH, "dataset_Brandon.csv"))

if LIMIT is not None:
    df_input = df_input.head(LIMIT)

CHECKPOINT_PATH = os.path.join(OUTPUT_PATH, "featextract_checkpoint.pkl")
SAVE_EVERY = 25

# resume from checkpoint if it exists
if os.path.exists(CHECKPOINT_PATH):
    with open(CHECKPOINT_PATH, "rb") as f:
        checkpoint = pickle.load(f)
    all_layer_texts = checkpoint["all_layer_texts"]
    all_refs = checkpoint["all_refs"]
    index_map = checkpoint["index_map"]
    start_idx = checkpoint["next_idx"]
    print(f"Resumed from checkpoint at question {start_idx}/{len(df_input)}")
else:
    all_layer_texts = []
    all_refs = []
    index_map = []
    start_idx = 0

for i, row in tqdm(df_input.iterrows(), total=len(df_input)):
    if i < start_idx:
        continue

    prompt = row["question"]
    feat = extract_features(prompt)

    for layer_idx, layer_text in enumerate(feat["layer_texts"]):
        all_layer_texts.append(layer_text)
        all_refs.append(feat["final_text"])

        index_map.append({
            "question_id": row["question_id"],
            "answer": row["answer"],
            "type": row["type"],
            "question": prompt,
            "layer": layer_idx,
            "feat": feat
        })

    # periodic checkpoint
    if (i + 1) % SAVE_EVERY == 0:
        with open(CHECKPOINT_PATH, "wb") as f:
            pickle.dump({
                "all_layer_texts": all_layer_texts,
                "all_refs": all_refs,
                "index_map": index_map,
                "next_idx": i + 1
            }, f)
        print(f"Checkpoint saved at question {i + 1}/{len(df_input)}")

# final save
with open(CHECKPOINT_PATH, "wb") as f:
    pickle.dump({
        "all_layer_texts": all_layer_texts,
        "all_refs": all_refs,
        "index_map": index_map,
        "next_idx": len(df_input)
    }, f)
print("feature extraction complete")

# chunked BERTScore so gpu don't die
CHUNK_SIZE = 256
P_all, R_all, F1_all = [], [], []

for start in tqdm(range(0, len(all_layer_texts), CHUNK_SIZE), desc="BERTScore"):
    end = min(start + CHUNK_SIZE, len(all_layer_texts))
    p, r, f1 = scorer.score(all_layer_texts[start:end], all_refs[start:end])
    P_all.append(p)
    R_all.append(r)
    F1_all.append(f1)

P = torch.cat(P_all)
R = torch.cat(R_all)
F1 = torch.cat(F1_all)

  0%|                                                                                             | 0/3 [00:00<?, ?it/s]

[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


[transformers] Both `max_new_tokens` (=100) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  0%|                                                                                             | 0/3 [13:43<?, ?it/s]

In [ ]:
# organize results into dataframes
rows_summary = []   # one row per question
rows_layers = []    # one row per (question, layer)

seen_qids = set()

for idx, item in enumerate(index_map):
    feat = item["feat"]

    qid = item["question_id"]

    if qid not in seen_qids:
        rows_summary.append({
            "question_id": qid,
            "type": item["type"],
            "question": item["question"],
            "answer": item["answer"],
            "final_text": feat["final_text"],
            "log_confidence": feat["log_confidence"],
            "normalized_log_confidence": feat["normalized_log_confidence"]
        })
        seen_qids.add(qid)

    rows_layers.append({
        "question_id": qid,
        "type": item["type"],
        "layer": item["layer"],
        "layer_text": all_layer_texts[idx],
        "precision": P[idx].item(),
        "recall": R[idx].item(),
        "f1": F1[idx].item()
    })

results_summary = pd.DataFrame(rows_summary)
results_layers = pd.DataFrame(rows_layers)

In [ ]:
# save results to drive
SUMMARY_PATH = os.path.join(OUTPUT_PATH, "results_summary.csv")
LAYERS_PATH = os.path.join(OUTPUT_PATH, "results_layers.csv")

results_summary.to_csv(SUMMARY_PATH, index=False)
results_layers.to_csv(LAYERS_PATH, index=False)

print(f"Saved summary to: {SUMMARY_PATH}")
print(f"Saved layer details to: {LAYERS_PATH}")

In [ ]:
# plot results
import matplotlib.pyplot as plt

plt.figure()

for qid in results_layers["question_id"].unique():
    subset = results_layers[results_layers["question_id"] == qid]
    plt.plot(subset["layer"], subset["f1"], label=f"QID {qid}")

plt.xlabel("Layer")
plt.ylabel("BERTScore F1")
plt.title("Layer-wise Semantic Alignment")
#plt.legend()
plt.show()

### Judge Sample Questions

In [ ]:
import pandas as pd, os, json

SUMMARY_PATH = os.path.join(OUTPUT_PATH, "results_summary.csv")
df = pd.read_csv(SUMMARY_PATH)

BATCH_SIZE = 50

BATCH_PROMPT_HEADER = """
You are a strict classification judge. For each numbered entry below, classify
what TYPE of reply the model gave. The three types are:
  - abstain : the model refuses, says it does not know, or declines
  - clarify : the model asks for more info / says the question is ambiguous
  - answer  : the model gives a direct, substantive answer

Return ONLY a JSON array of objects with keys "id" and "predicted_type".
Example output:
[{"id": 1, "predicted_type": "answer"}, {"id": 2, "predicted_type": "abstain"}]

Do NOT output anything else — no markdown fences, no explanation.

Entries:
"""

# build all batches
n_batches = (len(df) + BATCH_SIZE - 1) // BATCH_SIZE

for batch_idx in range(n_batches):
    start = batch_idx * BATCH_SIZE
    end = min(start + BATCH_SIZE, len(df))
    batch_df = df.iloc[start:end]

    entries = []
    for _, row in batch_df.iterrows():
        entries.append(
            f"     id: {row['question_id']}\n"
            f"     Question: {row['question']}\n"
            f"     Model reply: {row['final_text']}"
        )

    full_prompt = BATCH_PROMPT_HEADER + "\n\n".join(entries)

    print("=" * 70)
    print(f"BATCH {batch_idx + 1} / {n_batches}  (rows {start}–{end - 1})")
    print("=" * 70)
    print(full_prompt)
    print("=" * 70)
    print()

print(f"\nTotal rows: {len(df)}  |  Total batches: {n_batches}")
